In [14]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.functions import rand, floor, element_at, lit, array

In [15]:


# Criar uma sessão Spark
spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("Exercicio Intro") \
    .getOrCreate()

In [16]:

# Carregar o arquivo para um DataFrame chamado df_nomes
df_nomes = spark.read.csv("nomes_aleatorios.txt", header=False, inferSchema=True)

# Listar algumas linhas do DataFrame
df_nomes.show(5)


+------------------+
|               _c0|
+------------------+
|       Roy Fuhrman|
|Roosevelt Kirchner|
|   Milton Longoria|
|     Thomas Ringel|
|     John Mckinney|
+------------------+
only showing top 5 rows


In [17]:
df_nomes.printSchema()

root
 |-- _c0: string (nullable = true)



In [18]:
# Renomear a coluna para 'Nomes'
df_nomes = df_nomes.withColumnRenamed("_c0", "Nomes")


In [19]:

# Imprimir o esquema do DataFrame
df_nomes.printSchema()

root
 |-- Nomes: string (nullable = true)



In [20]:
# Mostrar 10 linhas do DataFrame
df_nomes.show(10)

+-------------------+
|              Nomes|
+-------------------+
|        Roy Fuhrman|
| Roosevelt Kirchner|
|    Milton Longoria|
|      Thomas Ringel|
|      John Mckinney|
|Christopher Timothy|
|       Lisa Hammond|
|       Olga Naranjo|
|     Thomas Padgett|
|      Samuel Silver|
+-------------------+
only showing top 10 rows


In [ ]:
escolaridades = ["Fundamental", "Medio", "Superior"]

escolaridades = ["Fundamental", "Medio", "Superior"]

df_nomes = df_nomes.withColumn(
    "Escolaridade",
    element_at(
        array(*[lit(e) for e in escolaridades]),
        (floor(rand() * len(escolaridades)) + 1).cast("int")
    )
)

df_nomes.show(10)

+-------------------+------------+
|              Nomes|Escolaridade|
+-------------------+------------+
|        Roy Fuhrman| Fundamental|
| Roosevelt Kirchner| Fundamental|
|    Milton Longoria| Fundamental|
|      Thomas Ringel|       Medio|
|      John Mckinney| Fundamental|
|Christopher Timothy| Fundamental|
|       Lisa Hammond|    Superior|
|       Olga Naranjo|       Medio|
|     Thomas Padgett|       Medio|
|      Samuel Silver| Fundamental|
+-------------------+------------+
only showing top 10 rows


In [22]:
paises = [
    "Argentina", "Bolívia", "Brasil", "Chile", "Colômbia", "Equador", "Guiana",
    "Paraguai", "Peru", "Suriname", "Uruguai", "Venezuela", "Guiana Francesa"
]

df_nomes = df_nomes.withColumn(
    "Pais",
    element_at(
        array(*[lit(p) for p in paises]),
        (floor(rand() * len(paises)) + 1).cast("int")
    )
)

df_nomes.show(10)

+-------------------+------------+---------------+
|              Nomes|Escolaridade|           Pais|
+-------------------+------------+---------------+
|        Roy Fuhrman| Fundamental|         Brasil|
| Roosevelt Kirchner| Fundamental|      Venezuela|
|    Milton Longoria| Fundamental|        Uruguai|
|      Thomas Ringel|       Medio|Guiana Francesa|
|      John Mckinney| Fundamental|      Argentina|
|Christopher Timothy| Fundamental|      Argentina|
|       Lisa Hammond|    Superior|        Bolívia|
|       Olga Naranjo|       Medio|       Suriname|
|     Thomas Padgett|       Medio|       Suriname|
|      Samuel Silver| Fundamental|       Colômbia|
+-------------------+------------+---------------+
only showing top 10 rows


In [24]:
df_nomes = df_nomes.withColumn(
    "AnoNascimento",
    (floor(rand() * (2010 - 1945 + 1)) + 1945).cast("int")
)

df_nomes.show(10)

+-------------------+------------+---------------+-------------+
|              Nomes|Escolaridade|           Pais|AnoNascimento|
+-------------------+------------+---------------+-------------+
|        Roy Fuhrman| Fundamental|         Brasil|         1958|
| Roosevelt Kirchner| Fundamental|      Venezuela|         1945|
|    Milton Longoria| Fundamental|        Uruguai|         1970|
|      Thomas Ringel|       Medio|Guiana Francesa|         1989|
|      John Mckinney| Fundamental|      Argentina|         1999|
|Christopher Timothy| Fundamental|      Argentina|         1998|
|       Lisa Hammond|    Superior|        Bolívia|         1958|
|       Olga Naranjo|       Medio|       Suriname|         1974|
|     Thomas Padgett|       Medio|       Suriname|         1965|
|      Samuel Silver| Fundamental|       Colômbia|         2008|
+-------------------+------------+---------------+-------------+
only showing top 10 rows


In [25]:
df_select = df_nomes.select("Nomes", "AnoNascimento").filter(df_nomes.AnoNascimento >= 2001)
df_select.show(10)

+-----------------+-------------+
|            Nomes|AnoNascimento|
+-----------------+-------------+
|    Samuel Silver|         2008|
|      Dora Newton|         2006|
|Stephanie Stewart|         2001|
|  Richard Bundage|         2010|
|  Deborah Lizotte|         2009|
|     Fawn Tallent|         2003|
|     Kristi Ellis|         2006|
|    Thomas Franks|         2006|
|Dorothy Delatorre|         2007|
|    Michelle Bost|         2002|
+-----------------+-------------+
only showing top 10 rows


In [27]:
# Registrar o DataFrame como uma tabela temporária
df_nomes.createOrReplaceTempView("pessoas")

# Executar a consulta SQL para selecionar nomes e ano de nascimento a partir de 2001
df_select_sql = spark.sql("""
    SELECT Nomes, AnoNascimento
    FROM pessoas
    WHERE AnoNascimento >= 2001
""")

df_select_sql.show(10)

+-----------------+-------------+
|            Nomes|AnoNascimento|
+-----------------+-------------+
|    Samuel Silver|         2008|
|      Dora Newton|         2006|
|Stephanie Stewart|         2001|
|  Richard Bundage|         2010|
|  Deborah Lizotte|         2009|
|     Fawn Tallent|         2003|
|     Kristi Ellis|         2006|
|    Thomas Franks|         2006|
|Dorothy Delatorre|         2007|
|    Michelle Bost|         2002|
+-----------------+-------------+
only showing top 10 rows


In [47]:
# Filtra os Millennials
millennials_df = df_nomes.select("Nomes", "AnoNascimento") \
    .filter((df_nomes.AnoNascimento >= 1980) & (df_nomes.AnoNascimento <= 1994))

# Cria um DataFrame com a contagem
count_df = millennials_df.agg({"AnoNascimento": "count"}) \
    .withColumnRenamed("count(AnoNascimento)", "Quantidade_Millennials")

# Mostra o resultado
count_df.show()

+----------------------+
|Quantidade_Millennials|
+----------------------+
|               2273315|
+----------------------+



In [39]:
# Registrar o DataFrame como uma tabela temporária
df_nomes.createOrReplaceTempView("pessoas")

# Contar o número de pessoas Millennials usando Spark SQL
resultado = spark.sql("""
    SELECT COUNT(*) AS Quantidade_Millennials
    FROM pessoas
    WHERE AnoNascimento BETWEEN 1980 AND 1994
""")

# Mostrar o resultado
resultado.show()

+----------------------+
|Quantidade_Millennials|
+----------------------+
|               2273315|
+----------------------+



In [34]:
df_geracoes_pais = spark.sql("""
    SELECT 
        Pais,
        CASE 
            WHEN AnoNascimento BETWEEN 1944 AND 1964 THEN 'Baby Boomers'
            WHEN AnoNascimento BETWEEN 1965 AND 1979 THEN 'Geração X'
            WHEN AnoNascimento BETWEEN 1980 AND 1994 THEN 'Millennials'
            WHEN AnoNascimento BETWEEN 1995 AND 2015 THEN 'Geração Z'
            ELSE 'Outra'
        END AS Geracao,
        COUNT(*) AS Quantidade
    FROM pessoas
    WHERE AnoNascimento BETWEEN 1944 AND 2015
    GROUP BY Pais, Geracao
    ORDER BY Pais ASC, Geracao ASC, Quantidade ASC
""")



In [35]:
df_geracoes_pais.show(n=df_geracoes_pais.count(), truncate=False)

+---------------+------------+----------+
|Pais           |Geracao     |Quantidade|
+---------------+------------+----------+
|Argentina      |Baby Boomers|233039    |
|Argentina      |Geração X   |175323    |
|Argentina      |Geração Z   |186309    |
|Argentina      |Millennials |174793    |
|Bolívia        |Baby Boomers|232690    |
|Bolívia        |Geração X   |175359    |
|Bolívia        |Geração Z   |186540    |
|Bolívia        |Millennials |174732    |
|Brasil         |Baby Boomers|233287    |
|Brasil         |Geração X   |174951    |
|Brasil         |Geração Z   |186492    |
|Brasil         |Millennials |174613    |
|Chile          |Baby Boomers|233158    |
|Chile          |Geração X   |174445    |
|Chile          |Geração Z   |186423    |
|Chile          |Millennials |175023    |
|Colômbia       |Baby Boomers|232465    |
|Colômbia       |Geração X   |174595    |
|Colômbia       |Geração Z   |185755    |
|Colômbia       |Millennials |174359    |
|Equador        |Baby Boomers|2333